# Replicação do MTST no ETTh1 (Colab, GPU)
Antes: *Runtime > Alterar tipo de ambiente de execução > GPU* e criar o Secret `GITHUB_TOKEN` (ícone de chave, com acesso ao notebook ligado).

In [ ]:
# Clona o repo privado com o token dos Secrets (o token não é gravado no notebook nem em .git/config)
import base64, os, subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
if not os.path.exists('/content/mtst-demanda-varejo'):
    subprocess.run(['git', '-c', f'http.extraheader=AUTHORIZATION: basic {auth}', 'clone',
                    'https://github.com/VanthuirMaia/mtst-demanda-varejo.git',
                    '/content/mtst-demanda-varejo'], check=True)
%cd /content/mtst-demanda-varejo
!git log --oneline -3

In [ ]:
# torch, pandas e numpy já vêm no Colab; falta só isto
!pip install -q einops yacs

In [ ]:
# ETTh1 do repositório público do ETDataset
!mkdir -p data/raw
!wget -q -O data/raw/ETTh1.csv https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv
!wc -l data/raw/ETTh1.csv

In [ ]:
import torch
assert torch.cuda.is_available(), 'Sem GPU: ative em Ambiente de execução > Alterar tipo'
print(torch.cuda.get_device_name(0))

In [ ]:
# 4 horizontes em sequência (cada um atualiza results/tables/etth1_resultados.csv)
for H in (96, 192, 336, 720):
    !python -u -m src.replicacao --horizonte {H}

In [ ]:
import pandas as pd
pd.read_csv('results/tables/etth1_resultados.csv')

In [ ]:
# Compacta resultados e logs para download
!zip -q -r etth1_resultados.zip results/tables/etth1_*
from google.colab import files
files.download('etth1_resultados.zip')